# Planetary observation satellite

This is a synthetic, count-only store-and-forward mission, **not** an orbital, attitude or flight-qualified model. The notebook uses the existing fmdtools propagation API and requires no external data or model weights.

```text
Copyright © 2024, United States Government, as represented by the Administrator
of the National Aeronautics and Space Administration. All rights reserved.

The "Fault Model Design tools - fmdtools version 2" software is licensed
under the Apache License, Version 2.0 (the "License"); you may not use this
file except in compliance with the License. You may obtain a copy of the
License at http://www.apache.org/licenses/LICENSE-2.0.

Unless required by applicable law or agreed to in writing, software distributed
under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR
CONDITIONS OF ANY KIND, either express or implied. See the License for the
specific language governing permissions and limitations under the License.
```


In [ ]:
from fmdtools.sim import propagate
from fmdtools_examples.planetary_observation.model_main import ObservationSatellite

model = ObservationSatellite()
result, history = propagate.nominal(model)
summary = {key.removeprefix("tend.classify."): value for key, value in result.items()}
summary


## Nominal mission timeline

Four acquisitions occur at minutes 1–4. The threshold selects the second and fourth; contacts at minutes 5 and 6 deliver both packets.


In [ ]:
import matplotlib.pyplot as plt

time = history["time"]
fig, ax = plt.subplots(figsize=(9, 4))
for label, key in [
    ("selected", "fxns.process.s.selected"),
    ("queued", "flows.buffer.s.queued"),
    ("downlinked", "fxns.downlink.s.downlinked"),
    ("dropped", "fxns.process.s.dropped"),
]:
    ax.step(time, history[key], where="post", label=label)
ax.set(xlabel="Mission time (minutes)", ylabel="Cumulative packets / backlog", title="Nominal mission counts")
ax.legend()
plt.show()
assert (history["fxns.process.s.selected"] == history["fxns.downlink.s.downlinked"] + history["flows.buffer.s.queued"] + history["fxns.process.s.dropped"]).all()


## Two operational variations

Compare a buffer with room for only one packet and a schedule with no ground contacts. These are **deterministic operational configurations**, not sampled faults or probabilistic risk estimates.


In [ ]:
for label, params in [
    ("buffer capacity 1", {"buffer_capacity": 1}),
    ("no contacts", {"contact_times": ()}),
]:
    changed_result, _ = propagate.nominal(ObservationSatellite(p=params))
    changed = {key.removeprefix("tend.classify."): value for key, value in changed_result.items()}
    print(label, {key: changed[key] for key in ("selected", "downlinked", "queued", "dropped")})
    assert changed["selected"] == changed["downlinked"] + changed["queued"] + changed["dropped"]


## Extension boundaries

A later increment can replace `ProcessObservation.select_observation` with a model adapter and add explicit failure modes, schedule perturbations and mission outcome measures. The current result provides no calibrated failure probability, classifier accuracy, downlink bitrate or orbital dynamics.
